# Step 4: Model Evaluation & Performance Review

This notebook evaluates the fine-tuned LLMs by testing them against a random subset of our dataset. It measures their ability to output valid JSON and their exact correctness across multiple CRM fields.

## 0. Setup Google Colab (Mount Drive)
Run this block to mount your Google Drive and enter the project directory.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
except ImportError:
    print("Not running in Google Colab, skipping drive mount.")

Mounted at /content/drive
/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor


In [ ]:
try:
    import google.colab
    !pip install -q condacolab
    import condacolab
    condacolab.install()
except ImportError:
    print("Not running in Colab. Skipping Conda setup.")

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"

    print("\n--- Installing Environment ---")
    !conda env update -n base -f environment.yml

    print("\n--- Installing Colab Unsloth Drivers ---")
    !pip install pandas
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "xformers<0.0.27" peft accelerate bitsandbytes
except ImportError:
    print("Not running in Colab. Skipping mount and environment update.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor

--- Installing Environment ---
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: - failed

SpecsConfigurationConflictError: Requested specs conflict with configured specs.
  requested specs: 
    - ffmpeg=8.0.1
    - pip
    - python=3.11.15
  pinned specs: 
    - cuda-version=12
    - python=3.12
    - python_abi=3.12[build=*cp312*]
Use 'conda config --show-sources' to look for 'pinned_specs' and 'track_features'
configuration parameters.  Pinned specs may also be defined in the file
/usr/local/conda-meta/pinned.



--- Installing Colab Unsloth Drivers ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-hfrnwcpb/unsloth_4415f7665a98406f905605fd259157ed
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-hfrnwcpb/unsloth_4415f7665a98406f905605fd259157ed
  Resolved https://github.com/unslothai/unsloth.git to commit 382683ebdc0b7ed55f53cd7698664164be843dab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 149.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 52.9 MB/s eta 0:00:00

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 MB 10.1 MB/s eta 0:00:00


## 1. Run Evaluator Pipeline
We will now feed 20 test transcripts into each model (Qwen3.5 2B, Qwen3.5 0.8B, Gemma-4 E4B) and compare their structured JSON responses directly against the Ground Truth created by Gemini.
This script directly tracks:
1. JSON Parsability (can it output valid JSON?)
2. Boolean Accuracy (Busy matching)
3. Exact String Matching (Sector, Scheduled At)
4. Mean Absolute Error (Interested, Rating)

In [ ]:
%cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
# !python pipeline/evaluator.py --models data/models/Qwen3.5-2B_lora data/models/Qwen3.5-0.8B_lora data/models/gemma-4-E4B_lora

# !python pipeline/evaluator.py --models Qwen/Qwen3.5-2B data/models/Qwen3.5-2B_lora Qwen/Qwen3.5-0.8B
!python pipeline/evaluator.py --models Qwen/Qwen3.5-2B Qwen/Qwen3.5-0.8B

/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W0523 04:26:43.307000 13442 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0523 04:26:43.356000 13442 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth Zoo will now patch everything to make training faster!
<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anythin

## 2. Visualize Results
Let's load the `evaluation_results.csv` and plot the performance story.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
    df = pd.read_csv("data/evaluation_results.csv")
    display(df)

    # Set up the plot grid
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")

    # 1. JSON Parsability
    sns.barplot(data=df, x="Model", y="Valid_JSON_%", ax=axes[0], palette="Blues_d")
    axes[0].set_title("Valid JSON Output Rate (%)")
    axes[0].set_ylim(0, 100)

    # 2. Boolean & String Matches
    df_melted = df.melt(id_vars="Model", value_vars=["Busy_Accuracy_%", "Sector_Match_%", "Schedule_Match_%"],
                        var_name="Metric", value_name="Score")
    sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", ax=axes[1])
    axes[1].set_title("Field Exact Match Accuracy (%)")
    axes[1].set_ylim(0, 100)

    # 3. Mean Absolute Error (MAE) for Ratings
    df_mae = df.melt(id_vars="Model", value_vars=["Interested_MAE", "Rating_MAE"],
                     var_name="Metric", value_name="MAE")
    sns.barplot(data=df_mae, x="Model", y="MAE", hue="Metric", ax=axes[2])
    axes[2].set_title("Rating Error (MAE - Lower is Better)")

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("evaluation_results.csv not found! Run the evaluator cell first.")